# 05.1 Time-Series Live Scoring

Scoring de mercados activos con `PriceSequenceGRU`, más una evaluación backtest-style sobre resueltos.

**Objetivo**
- cargar el checkpoint TS ya entrenado
- puntuar mercados activos usando solo `price_history`
- generar señales con las mismas reglas del baseline
- revisar cómo se comporta el score sobre una muestra de resueltos

**Prerequisito**
Este notebook asume que ya existe `data/models/ts_gru/best_ts_gru_model.pt`.

## 1. Setup y carga de artefactos

Trabajamos con el snapshot local de `data/raw/`, sin depender de descargas nuevas. Eso mantiene la comparación contra el baseline en el mismo corte de datos.

In [ ]:
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.model.ts_architecture import PriceSequenceGRU
from src.scoring.ts_scorer import score_active_markets_ts, score_resolved_markets_ts
from src.scoring.signals import generate_signals

logging.getLogger('src.scoring.ts_scorer').setLevel(logging.WARNING)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

cfg = load_config(str(ROOT / 'config' / 'config.yaml'))
RAW = ROOT / 'data' / 'raw'
MODELS_DIR = ROOT / cfg['ts_training']['save_dir']
FIGURES = ROOT / 'figures'

model_path = MODELS_DIR / 'best_ts_gru_model.pt'
assert model_path.exists(), f'Modelo TS no encontrado: {model_path}'


## 2. Cliente local y carga de datos

El cliente local reproduce la interfaz del fetcher real, pero usa archivos cacheados. Así el notebook es reproducible y no depende de la red.

In [ ]:
class LocalClient:
    def __init__(self, markets):
        self.markets = markets
    def get_all_active_markets(self, max_markets=1000):
        return self.markets[:max_markets]
    def parse_market(self, market):
        return market

with open(RAW / 'active_markets.json') as f:
    active_markets = json.load(f)
with open(RAW / 'resolved_markets.json') as f:
    resolved_markets = json.load(f)
with open(RAW / 'price_histories.json') as f:
    price_histories = json.load(f)

model = PriceSequenceGRU(
    input_dim=cfg['ts_model']['input_dim'],
    hidden_dim=cfg['ts_model']['hidden_dim'],
    num_layers=cfg['ts_model']['num_layers'],
    dropout=cfg['ts_model']['dropout'],
    task='classification',
)
model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=True))
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

client = LocalClient(active_markets)
print(f'Mercados activos cacheados: {len(active_markets):,}')
print(f'Price histories cacheados: {len(price_histories):,}')
print(f'Dispositivo: {device}')


## 3. Scoring de mercados activos

El modelo TS usa solo historia de precios. Si un mercado no tiene suficientes puntos válidos, se omite. Después aplicamos exactamente las mismas reglas de señal que en el baseline.

In [ ]:
df_scores = score_active_markets_ts(
    model,
    client,
    price_histories=price_histories,
    fetch_missing_history=False,
    top_k=len(active_markets),
    max_markets=len(active_markets),
    seq_len=cfg['ts_data']['seq_len'],
    min_points=cfg['ts_data']['min_points'],
)

df_scores = generate_signals(
    df_scores,
    buy_threshold=cfg['scoring']['buy_threshold'],
    strong_buy_threshold=cfg['scoring']['strong_buy_threshold'],
    min_liquidity=cfg['scoring']['min_liquidity'],
    min_volume_24h=cfg['scoring']['min_volume_24h'],
    max_spread=cfg['scoring']['max_spread'],
)

print(f'Mercados puntuados por TS-GRU: {len(df_scores):,}')
print(df_scores[['signal', 'model_score', 'price_yes', 'sequence_length']].head())


## 4. Lectura rápida de señales

Esta tabla sirve para ver si el modelo está concentrando score en pocos mercados o si todo quedó en `HOLD`.

In [ ]:
signal_counts = df_scores['signal'].value_counts().reindex(['STRONG BUY', 'BUY', 'HOLD'], fill_value=0)
top_signals = df_scores[df_scores['signal'] != 'HOLD'].head(20)

print('Distribución de señales TS:')
for signal, count in signal_counts.items():
    print(f'  {signal:12s}: {count:,} ({100 * count / len(df_scores):.1f}%)')

print('\nTop señales TS:')
for idx, (_, row) in enumerate(top_signals.iterrows(), 1):
    print(f"{idx:>2}. {row['signal']:12s} score={row['model_score']:.3f} price={row['price_yes']:.2f} len={int(row['sequence_length'])} | {row['question'][:70]}")


## 5. Distribución de scores y señales

Dos chequeos rápidos útiles:
- cómo se reparte el `model_score` sobre el universo activo
- cuántos mercados sobreviven a los filtros operativos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(df_scores['model_score'], bins=40, color=PALETTE[0], alpha=0.85)
ax.axvline(cfg['scoring']['buy_threshold'], color='gray', linestyle='--', lw=1.5, label='BUY threshold')
ax.axvline(cfg['scoring']['strong_buy_threshold'], color='black', linestyle='--', lw=1.5, label='STRONG BUY threshold')
ax.set_title('Distribución de TS model_score')
ax.set_xlabel('model_score')
ax.legend(fontsize=9)

ax = axes[1]
ax.bar(signal_counts.index, signal_counts.values, color=[PALETTE[3], PALETTE[2], PALETTE[0]], alpha=0.85)
ax.set_title('Señales de live scoring — TS-GRU')
ax.set_ylabel('Mercados')

plt.tight_layout()
plt.savefig(FIGURES / 'ts_live_signals.png', bbox_inches='tight')
plt.show()


## 6. Evaluación backtest-style en resueltos

No es un backtest de ejecución real, pero sí una lectura útil de consistencia: truncamos la serie al snapshot histórico, scoreamos y medimos win rate por señal.

In [ ]:
BACKTEST_SAMPLE = 2000
df_bt = score_resolved_markets_ts(
    model,
    resolved_markets[:BACKTEST_SAMPLE],
    price_histories=price_histories,
    snapshot_offset_days=cfg['ts_data']['snapshot_offset_days'],
    seq_len=cfg['ts_data']['seq_len'],
    min_points=cfg['ts_data']['min_points'],
    device=device,
)
df_bt = generate_signals(
    df_bt,
    buy_threshold=cfg['scoring']['buy_threshold'],
    strong_buy_threshold=cfg['scoring']['strong_buy_threshold'],
    min_liquidity=cfg['scoring']['min_liquidity'],
    min_volume_24h=cfg['scoring']['min_volume_24h'],
    max_spread=cfg['scoring']['max_spread'],
)

print(f'Muestra backtest TS: {len(df_bt):,} mercados')
for signal in ['STRONG BUY', 'BUY', 'HOLD']:
    sub = df_bt[df_bt['signal'] == signal]
    if len(sub) == 0:
        continue
    print(f"  {signal:12s}: {len(sub):>4,} | win rate Yes = {100 * sub['resolved'].mean():.1f}%")


## 7. Diagnóstico de calibración por señal

La idea aquí es ver si `STRONG BUY` realmente concentra un win rate mayor que `BUY`, y ambos mayor que la base de la muestra.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sig_order = ['STRONG BUY', 'BUY', 'HOLD']
sig_colors = [PALETTE[3], PALETTE[2], PALETTE[0]]
wr_vals, counts = [], []
for sig in sig_order:
    sub = df_bt[df_bt['signal'] == sig]
    wr_vals.append(sub['resolved'].mean() if len(sub) > 0 else 0)
    counts.append(len(sub))
bars = ax.bar(sig_order, wr_vals, color=sig_colors, alpha=0.85)
for bar, wr, cnt in zip(bars, wr_vals, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{100*wr:.1f}%\n(n={cnt:,})', ha='center', fontsize=9)
ax.axhline(df_bt['resolved'].mean(), color='black', linestyle='--', lw=1.5, label=f'Base rate ({100*df_bt["resolved"].mean():.1f}%)')
ax.set_ylim(0, 1)
ax.set_ylabel('Win rate (% Yes)')
ax.set_title('Win rate por señal — TS-GRU')
ax.legend(fontsize=9)

ax = axes[1]
df_bt['score_bin'] = pd.cut(df_bt['model_score'], bins=10)
wr_by_bin = df_bt.groupby('score_bin', observed=True)['resolved'].agg(['mean', 'count'])
centers = [interval.mid for interval in wr_by_bin.index]
ax.bar(range(len(wr_by_bin)), wr_by_bin['mean'], color=PALETTE[0], alpha=0.8)
ax.axhline(df_bt['resolved'].mean(), color='red', linestyle='--', lw=1.5, label='Base rate')
ax.set_xticks(range(len(wr_by_bin)))
ax.set_xticklabels([f'{c:.2f}' for c in centers], rotation=45, fontsize=8)
ax.set_title('Win rate vs score — TS-GRU')
ax.set_xlabel('Score bin')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / 'ts_live_backtest.png', bbox_inches='tight')
plt.show()


## 8. Qué mirar antes de comparar con el baseline

Este notebook queda “bien” si ves algo como esto:
- cobertura razonable en activos
- algunas señales distintas de `HOLD`
- win rate creciente al subir el score o la fuerza de la señal

Con eso, el siguiente paso es `05_2_model_comparison.ipynb`.